In [1]:
import math

from Util.Problems import Problem, solution
from Util.math_functions import get_prime_factors, prime_sieve, get_triangular_root, get_triangular_number


class P012(Problem):
    number = 12
    title = "Highly Divisible Triangular Number"
    description = """<p>The sequence of triangle numbers is generated by adding the natural numbers. So the $7$<sup>th</sup> triangle number would be $1 + 2 + 3 + 4 + 5 + 6 + 7 = 28$. The first ten terms would be:
$$1, 3, 6, 10, 15, 21, 28, 36, 45, 55, \\dots$$</p><p>Let us list the factors of the first seven triangle numbers:</p>
$$\\begin{align}
\\mathbf 1 &\\colon 1\\\\
\\mathbf 3 &\\colon 1,3\\\\
\\mathbf 6 &\\colon 1,2,3,6\\\\
\\mathbf{10} &\\colon 1,2,5,10\\\\
\\mathbf{15} &\\colon 1,3,5,15\\\\
\\mathbf{21} &\\colon 1,3,7,21\\\\
\\mathbf{28} &\\colon 1,2,4,7,14,28
\\end{align}$$
<p>We can see that $28$ is the first triangle number to have over five divisors.</p><p>What is the value of the first triangle number to have over five hundred divisors?</p>"""
    divisors = 500

In [2]:
p = P012()
p.describe()

## Problem 12: Highly Divisible Triangular Number

<p>The sequence of triangle numbers is generated by adding the natural numbers. So the $7$<sup>th</sup> triangle number would be $1 + 2 + 3 + 4 + 5 + 6 + 7 = 28$. The first ten terms would be:
$$1, 3, 6, 10, 15, 21, 28, 36, 45, 55, \dots$$</p><p>Let us list the factors of the first seven triangle numbers:</p>
$$\begin{align}
\mathbf 1 &\colon 1\\
\mathbf 3 &\colon 1,3\\
\mathbf 6 &\colon 1,2,3,6\\
\mathbf{10} &\colon 1,2,5,10\\
\mathbf{15} &\colon 1,3,5,15\\
\mathbf{21} &\colon 1,3,7,21\\
\mathbf{28} &\colon 1,2,4,7,14,28
\end{align}$$
<p>We can see that $28$ is the first triangle number to have over five divisors.</p><p>What is the value of the first triangle number to have over five hundred divisors?</p>

### Solution notes
This loops through all triangle numbers and gets their prime factors. Using these, it calculates their factors using the following maths:

Any number n can be written as a prime factorisation:

$n = p_1^{e_1} \times p_2^{e_2} \times [...] \times p_n^{e_n} $

To determine the factors of such a number, each prime factor can be used 0 times, 1 time, etc up to e times to create a factor of n. Using combinatorics, this gives us a total number of factors of:

$(e_1 + 1)\times(e_2 + 1)\times[...]\times(e_n + 1)$

This is due to the fact that each factor can also not be used, which is why the +1 is needed. 1 and n are also considered factors, but this is also how the problem is stated.

In [3]:
@solution(P012, first=True, make_fast=True, warmup_args=(P012.divisors,))
def prime_factorisation(divisors):
    i = 1
    triangle_number = 1
    while True:
        prime_factors = get_prime_factors(triangle_number)
        amount_of_factors = 1

        number_of_prime_factors = len(prime_factors)
        index = 0
        while index < number_of_prime_factors:
            exponent = 1
            while index + exponent < number_of_prime_factors and prime_factors[index + exponent] == prime_factors[index]:
                exponent += 1
            amount_of_factors *= ( exponent + 1 )
            index += exponent

        if amount_of_factors > divisors:
            return triangle_number
        i += 1
        triangle_number += i
    return None

In [4]:
p.test_all(repeats=1)

76576500 found in 43.809800 ms by prime_factorisation (first)


This is how you would brute force this problem, but this is so extremely slow I have never even run it.

In [5]:
# @solution(P012, make_fast=True, warmup_args=(P012.divisors,))
# def brute_force(divisors):
#     i = 1
#     triangle_number = 1
#     while True:
#         factors = 0
#         for factor in range(1, triangle_number + 1):
#             if triangle_number % factor == 0:
#                 factors += 1
#         if factors > divisors:
#             return triangle_number
#         i += 1
#         triangle_number += i
#     return None

Using the logic from the first solution, we can determine the smallest overall number with n factors, by taking the prime factorisation of n, sorting it from largest to smallest, and applying those factors - 1 to the primes sorted smallest to largest.
In our example:

$500 = 5 \times 5 \times 5 \times 2 \times 2 $

Therefore, the smallest overall number with 500 factors is:

$2^4 \times 3^4 \times 5^4 \times 7^1 \times 11^1 = 62.370.000$

By calculating this lower bound, we can start checking triangular numbers from there and save a lot of time. To use this fairly tough, this bound will be calculated in the function as well, so this can be used for any number of divisors. Without that addition, this would feel like precomputing which I do not want to do in project Euler.

In [6]:
@solution(P012, best=True, make_fast=True, warmup_args=(P012.divisors,))
def lower_bound_factorisation(divisors):
    divisors_prime_factors = get_prime_factors(divisors)
    num_divisor_primes = len(divisors_prime_factors)
    lower_bound = 1
    first_n_primes = [2, 3, 5, 7, 11, 13, 17, 19, 23]
    for i in range(num_divisor_primes, 0, -1):
        lower_bound *= (first_n_primes[num_divisor_primes - i] ** (divisors_prime_factors[i - 1] - 1))

    # Now we start looping through the triangle numbers starting at the closest n to this lower bound
    n = math.floor(get_triangular_root(lower_bound))
    triangle_number = get_triangular_number(n)
    while True:
        prime_factors = get_prime_factors(triangle_number)
        amount_of_factors = 1

        number_of_prime_factors = len(prime_factors)
        index = 0
        while index < number_of_prime_factors:
            exponent = 1
            while index + exponent < number_of_prime_factors and prime_factors[index + exponent] == prime_factors[index]:
                exponent += 1
            amount_of_factors *= ( exponent + 1 )
            index += exponent

        if amount_of_factors > divisors:
            return triangle_number
        n += 1
        triangle_number += n
    return None

In [7]:
p.test_all()

76576500 found in 6.598194 ms by lower_bound_factorisation (best)
76576500 found in 42.419549 ms by prime_factorisation (first)
